# ML Challenge 2026 — Business Entity Resolution (Kaggle)

One switch in the setup cell:
- `RUN_TEST = False`: dev loop. Train split only; ends with val macro F0.5.
- `RUN_TEST = True`: full run. Also prepares test, runs sharded inference and writes both
  submission TSVs. Use **Save Version -> Save & Run All**.

Accelerator: None (CPU). Stages: normalize -> block (A-H, fair slots) -> features
(DuckDB + rapidfuzz cpdist) -> LightGBM + isotonic -> expected-F0.5 sets -> exclusivity.

In [ ]:
!pip install -q rapidfuzz duckdb pyarrow indic-transliteration lightgbm scikit-learn

In [ ]:
import glob, json, math, os, re, shutil, subprocess, sys, time, unicodedata
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path

import warnings
import duckdb, numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler
# Imported here so a missing package fails now, not silently inside Stage 0.
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

warnings.filterwarnings("ignore", message="fetch_record_batch")

RUN_TEST = False   # True -> test Stage 0 + sharded inference + submission files

DATA_DIR = Path("/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset")
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/amazon-ml-challenge-2026/dataset")
WORK_DIR = Path("/kaggle/working/work")
OUTPUT_DIR = Path("/kaggle/working/output")
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = [("train", 1, "train_source1.tsv"), ("train", 2, "train_source2.tsv"),
           ("train", 3, "train_source3.tsv"), ("test", 1, "test_source1.tsv"),
           ("test", 2, "test_source2.tsv"), ("test", 3, "test_source3.tsv")]
GROUND_TRUTH = "train_ground_truth.tsv"
SOURCE_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
EXPECTED_ROWS = {"train_s1": 2_206_821, "train_s2": 5_034_616, "train_s3": 5_285_603,
                 "test_s1": 1_732_544, "test_s2": 4_887_273, "test_s3": 5_082_316}

# Bump whenever normalize_name / normalize_address change: Stage 0 then rebuilds.
NORM_VERSION = 3

CONFIG = dict(
    # Stage 0
    prepare_limit=None,
    prepare_splits=("train", "test") if RUN_TEST else ("train",),
    workers=max(1, (os.cpu_count() or 4) - 1),

    # Stage 1
    s1_sample=50,            # per-mille of train S1 used to train/validate (50 = 5%)
    rare_per_s1=3, max_token_df=1000, skel_max_df=1000,
    addr_rare_per_s1=4, addr_max_df=300,
    pair_tokens=4, pair_per_s1=6, pair_max_df=2000,
    key_cap=300, scheme_cap=100, top_k=100,
    pair_budget=60_000_000,
    # scheme G: house number + street word
    g_tokens=4, g_per_s1=6, g_max_df=100, g_cap=20,
    # scheme H: name word x address word
    h_name_tokens=3, h_addr_tokens=3, h_per_s1=6, h_max_df=50, h_cap=20,

    # Kaggle: 4 vCPU, ~30 GB RAM, ~19.5 GB disk
    threads=4, mem_gb=14, spill_gb=8,
    test_shards=24,
)

print(f"data   {DATA_DIR}\nwork   {WORK_DIR}\noutput {OUTPUT_DIR}")
print(f"cpus {os.cpu_count()}  free disk {shutil.disk_usage(WORK_DIR).free / 2**30:.1f} GB  RUN_TEST={RUN_TEST}")

### Raw file check
`train_source1.tsv`'s header starts with a bare tab, so we always pass `names=`.

In [ ]:
for split, src, name in SOURCES:
    path = DATA_DIR / split / name
    with open(path, "rb") as fh:
        header = fh.readline().decode().rstrip("\n")
    flag = "  <-- unnamed first column!" if header.startswith("\t") else ""
    print(f"{split}/s{src}: {path.stat().st_size / 2**20:7.1f} MB  header={header!r}{flag}")

## EDA

In [ ]:
gt = pd.read_csv(DATA_DIR / "train" / GROUND_TRUTH, sep="\t",
                 dtype=str, keep_default_na=False, na_values=[])
empty = gt["matched_entity_ids"] == ""
counts = gt.loc[~empty, "matched_entity_ids"].str.count(",") + 1
print(f"S1 entities          {len(gt):,}")
print(f"singletons           {empty.sum():,} ({100 * empty.mean():.2f}%)")
print(f"mean matches/entity  {counts.sum() / len(gt):.2f}   max {counts.max()}")

mentions = gt.loc[~empty, "matched_entity_ids"].str.split(",").explode()
print("EXCLUSIVE: each S2/S3 record serves at most one S1 entity"
      if len(mentions) == mentions.nunique() else "NOT exclusive")
for split, src, name in [("train", 1, "train_source1.tsv"), ("test", 1, "test_source1.tsv")]:
    c = pd.read_csv(DATA_DIR / split / name, sep="\t", header=0, names=SOURCE_COLUMNS,
                    usecols=["country"], dtype=str)["country"].value_counts()
    print(f"\n{split} S1 countries:\n{c.to_string()}")

## Stage 0 — normalization

In [ ]:
_INDIC = re.compile(r"[ऀ-ൿ]")
_indic_run_re = re.compile(r"([ऀ-ൿ]+)")
_SCRIPT_BLOCKS = ((0x0900, 0x097F, "DEVANAGARI"), (0x0980, 0x09FF, "BENGALI"),
                  (0x0A00, 0x0A7F, "GURMUKHI"), (0x0A80, 0x0AFF, "GUJARATI"),
                  (0x0B00, 0x0B7F, "ORIYA"), (0x0B80, 0x0BFF, "TAMIL"),
                  (0x0C00, 0x0C7F, "TELUGU"), (0x0C80, 0x0CFF, "KANNADA"),
                  (0x0D00, 0x0D7F, "MALAYALAM"))

def _script_of(run):
    cp = ord(run[0])
    for lo, hi, name in _SCRIPT_BLOCKS:
        if lo <= cp <= hi:
            return name
    return "DEVANAGARI"

_DEVA_PHRASE_MAP = {
    "प्राइवेट": "private", "प्रा": "private", "लिमिटेड": "limited", "लिमिडेट": "limited",
    "लि": "limited", "कंपनी": "company", "कम्पनी": "company", "एंड": "and", "एण्ड": "and",
    "इंडिया": "india", "भारत": "india", "सर्विसेज": "services", "सर्विस": "service",
    "इंटरप्राइजेज": "enterprises", "एंटरप्राइजेज": "enterprises", "उद्योग": "udyog",
    "ट्रेडर्स": "traders", "टेक्नोलॉजीज": "technologies", "सॉल्यूशंस": "solutions",
    "इंडस्ट्रीज": "industries", "मार्केटिंग": "marketing", "फाइनेंस": "finance",
    "स्टोर्स": "stores", "स्टोर": "store",
}
_deva_phrase_re = re.compile("|".join(sorted(map(re.escape, _DEVA_PHRASE_MAP), key=len, reverse=True)))

_IAST_FIXUPS = (("ph", "f"), ("mg", "ng"))
_TAMIL_FIXUPS = (("dh", "t"), ("gh", "k"), ("bh", "p"))
_TRANSLIT_SUFFIX_RULES = (
    (re.compile(r"^limi"), "limited"),
    (re.compile(r"^(?:praiv|piraiv|privat|praiw)"), "private"),
    (re.compile(r"^(?:ailaail|elel|elail)"), "llp"),
    (re.compile(r"^(?:korpor|corpor)"), "corporation"),
    (re.compile(r"^(?:kampani|kampeni|kampan)"), "company"),
)

def fold_ascii(text):
    """Strip diacritics. Safe only AFTER transliteration."""
    return unicodedata.normalize("NFKD", text).encode("ASCII", "ignore").decode("ascii")

def _fix_indic_word(word, script):
    word = word.replace("ṃ", "n").replace("ṁ", "n")   # anusvara, before folding turns it into m
    word = fold_ascii(word).lower()
    if script == "TAMIL":
        for a, b in _TAMIL_FIXUPS:
            word = word.replace(a, b)
    for a, b in _IAST_FIXUPS:
        word = word.replace(a, b)
    if script == "DEVANAGARI" and len(word) > 3 and word.endswith("a"):
        word = word[:-1]
    for pattern, canon in _TRANSLIT_SUFFIX_RULES:
        if pattern.match(word):
            return canon
    return word

def _transliterate_indic(text):
    """Indic -> Latin, run by run, so Latin words already present are untouched."""
    text = _deva_phrase_re.sub(lambda m: " " + _DEVA_PHRASE_MAP[m.group(0)] + " ", text)
    if not _INDIC.search(text):
        return text
    out = []
    for i, part in enumerate(_indic_run_re.split(text)):
        if i % 2 == 0:
            out.append(part)
            continue
        script = _script_of(part)
        try:
            latin = transliterate(part, getattr(sanscript, script), sanscript.IAST)
        except Exception:
            out.append(part)
            continue
        out.append(" ".join(_fix_indic_word(w, script) for w in latin.split()))
    return "".join(out)

In [ ]:
LEGAL_SUFFIX_CANON = {
    "corporation": "corp", "corp": "corp", "corpn": "corp",
    "incorporated": "inc", "inc": "inc",
    "private": "pvt", "pvt": "pvt", "pte": "pvt",
    "limited": "ltd", "ltd": "ltd", "ltda": "ltd",
    "llp": "llp", "llc": "llc", "lp": "lp",
    "company": "co", "co": "co", "cos": "co",
    "sarl": "sarl", "sas": "sas", "sasu": "sas", "sa": "sa", "eurl": "eurl", "snc": "snc",
    "gmbh": "gmbh", "plc": "plc", "ag": "ag", "bv": "bv", "nv": "nv",
    "partnership": "partnership", "associates": "associates", "assoc": "associates",
    "holdings": "holdings", "group": "group", "trust": "trust", "society": "society",
}
NOISE_TOKENS = {"and", "the", "of", "a", "an"}
ADDR_ABBREV = {
    "rd": "road", "road": "road", "st": "street", "str": "street", "street": "street",
    "ave": "avenue", "av": "avenue", "avenue": "avenue", "blvd": "boulevard",
    "boulevard": "boulevard", "bd": "boulevard", "ln": "lane", "lane": "lane",
    "dr": "drive", "drive": "drive", "ct": "court", "court": "court", "pl": "place",
    "place": "place", "sq": "square", "square": "square", "hwy": "highway",
    "highway": "highway", "pkwy": "parkway", "parkway": "parkway", "apt": "apartment",
    "apartment": "apartment", "ste": "suite", "suite": "suite", "bldg": "building",
    "building": "building", "flr": "floor", "fl": "floor", "floor": "floor",
    "nr": "near", "near": "near", "opp": "opposite", "opposite": "opposite",
    "n": "north", "north": "north", "s": "south", "south": "south", "e": "east",
    "east": "east", "w": "west", "west": "west", "ne": "northeast", "nw": "northwest",
    "se": "southeast", "sw": "southwest", "marg": "marg", "nagar": "nagar",
    "colony": "colony", "sector": "sector", "phase": "phase", "block": "block",
    "gali": "gali", "chowk": "chowk", "po": "postoffice", "ps": "policestation",
    "dist": "district", "tq": "taluk", "taluk": "taluk", "tehsil": "taluk",
    "r": "rue", "rue": "rue",
}
ADDR_NOISE = {"apartment", "suite", "floor", "building", "unit", "no", "number",
              "near", "opposite", "postoffice", "policestation"}

_word_re = re.compile(r"[a-z0-9]+")
_amp_re = re.compile(r"&amp;|&#38;|&")
_zerowidth_re = re.compile(r"[​-‍﻿]")
_acronym_re = re.compile(r"\b[a-z](?:\.[a-z])+\.?")
_pin6_re, _zip5_re = re.compile(r"\b\d{6}\b"), re.compile(r"\b\d{5}\b")
_LEET = str.maketrans({"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "6": "g", "7": "t", "8": "b"})

def _unleet(tok):
    """'f0od'->'food', '6lobal'->'global'. Only words that MIX letters and digits."""
    if any(ch.isdigit() for ch in tok) and any(ch.isalpha() for ch in tok):
        return tok.translate(_LEET)
    return tok

def _pre_clean(raw):
    if raw is None:
        return ""
    text = str(raw)
    if not text or text == "nan":
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = _zerowidth_re.sub("", text)
    if _INDIC.search(text):
        text = _transliterate_indic(text)
    text = _amp_re.sub(" and ", text)
    text = fold_ascii(text).lower()
    return _acronym_re.sub(lambda m: m.group(0).replace(".", ""), text)

@dataclass(slots=True)
class NameParts:
    core: list = field(default_factory=list)
    suffix: list = field(default_factory=list)
    key: str = ""
    @property
    def text(self): return " ".join(self.core)

def normalize_name(raw):
    text = _pre_clean(raw)
    if not text:
        return NameParts()
    toks = _word_re.findall(text)
    skip = set()   # "... id 37752" is a generated record number, not part of the name
    for j, tok in enumerate(toks):
        if tok == "id" and j + 1 < len(toks) and toks[j + 1].isdigit():
            skip.update((j, j + 1))
    core, suffix = [], []
    for j, tok in enumerate(toks):
        if j in skip:
            continue
        tok = _unleet(tok)
        canon = LEGAL_SUFFIX_CANON.get(tok)
        if canon is not None:
            if canon not in suffix:
                suffix.append(canon)
            continue
        if tok in NOISE_TOKENS:
            continue
        core.append(tok)
    if not core and suffix:
        core = list(suffix)
    return NameParts(core, suffix, " ".join(sorted(set(core))))

@dataclass(slots=True)
class AddressParts:
    tokens: list = field(default_factory=list)
    numbers: list = field(default_factory=list)
    postcode: str = ""
    is_empty: bool = True
    @property
    def text(self): return " ".join(self.tokens)

def normalize_address(raw):
    text = _pre_clean(raw)
    if not text.strip():
        return AddressParts()
    m = _pin6_re.search(text) or _zip5_re.search(text)
    postcode = m.group(0) if m else ""
    tokens, numbers = [], []
    for tok in _word_re.findall(text):
        if tok == postcode:
            continue
        if tok.isdigit() or (any(c.isdigit() for c in tok) and any(c.isalpha() for c in tok)):
            if tok not in numbers:
                numbers.append(tok)
            continue
        tok = ADDR_ABBREV.get(tok, tok)
        if tok in NOISE_TOKENS or tok in ADDR_NOISE:
            continue
        tokens.append(tok)
    return AddressParts(tokens, numbers, postcode, False)

### Self-test — the Stage 0 gate (raises on failure)

In [ ]:
FUZZY_MIN = 60
NAME_CASES = [
    ("Ram Marketing Private Limited", "राम मार्केटिंग प्राइवेट लिमिटेड", "key"),
    ("मॉडर्न फाइनेंस", "Modern Finance", "fuzzy"),
    ("श्री साई इंफ्राटेक", "Shri Sai Infratech", "fuzzy"),
    ("Smith & Sons Corp", "Smith and Sons Corporation", "key"),
    ("Smith & Sons", "Smith Sons", "key"),
    ("Café Déjà Inc", "Cafe Deja Incorporated", "key"),
    ("Acme Traders Pvt Ltd", "Traders Acme Private Limited", "key"),
    ("Marina Ecole France Sarl", "Marina Ecole France S.A.R.L.", "key"),
    ("Zephay Labs Inc", "Zephyr Labs Inc", "differ"),
    ("Prime Money", "Prime Motors", "differ"),
]
INDIC_SAMPLES = [
    ("Devanagari", "राम मार्केटिंग प्राइवेट लिमिटेड"), ("Devanagari", "मॉडर्न फाइनेंस"),
    ("Malayalam", "സിൽവർ കൺസൾട്ടൻസി പ്രൈവറ്റ് ലിമിറ്റഡ്"),
    ("Gujarati", "આલ્ફા ફાઉન્ડેશન પ્રાઇવેટ લિમિટેડ"),
    ("Kannada", "ರಾಮ್ ಬಿಸಿನೆಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್"),
    ("Tamil", "ஸ்மார்ட் எக்ஸ்போர்ட்ஸ் பிரைவேட் லிமிடெட்"),
    ("Bengali", "রেড টেক প্রাইভেট লিমিটেড"),
    ("Telugu", "లోటస్ మార్కెటింగ్ ప్రైవేట్ లిమిటెడ్"),
    ("Gurmukhi", "ਸ਼ਿਵਮ ਗੋਲਡਨ ਪ੍ਰੋਡਿਊਸਰ ਐਲਐਲਪੀ"), ("Odia", "ସୁପର୍ ମ୍ୟାନେଜମେଣ୍ଟ୍"),
]

fails = 0
print("=== name normalization ===")
for a, b, mode in NAME_CASES:
    na, nb = normalize_name(a), normalize_name(b)
    same = na.key == nb.key and na.key != ""
    sim = fuzz.token_set_ratio(na.text, nb.text)
    ok = same if mode == "key" else (sim >= FUZZY_MIN if mode == "fuzzy" else not same)
    fails += not ok
    print(f"  [{'ok ' if ok else 'FAIL'}] {mode:6s} {na.key!r} | {nb.key!r}  sim={sim:.0f}")
print("\n=== non-empty guarantee, all 9 Indic scripts ===")
for script, raw in INDIC_SAMPLES:
    p = normalize_name(raw)
    fails += not p.core
    print(f"  [{'ok ' if p.core else 'FAIL'}] {script:11s} core={p.core} suffix={p.suffix}")
print(f"\n{'PASS' if fails == 0 else f'{fails} FAILURE(S)'}")
if fails:
    raise RuntimeError(f"Stage 0 self-test: {fails} failure(s). Do not run Stage 0.")

### Run Stage 0 (skips sources already built at this NORM_VERSION)

In [ ]:
import multiprocessing as mp

SCHEMA = pa.schema([
    ("entity_id", pa.string()), ("country", pa.string()), ("source", pa.int8()),
    ("name_key", pa.string()), ("name_text", pa.string()),
    ("name_tokens", pa.list_(pa.string())), ("name_suffix", pa.string()),
    ("n_core", pa.int16()), ("addr_text", pa.string()),
    ("addr_tokens", pa.list_(pa.string())), ("addr_nums", pa.list_(pa.string())),
    ("postcode", pa.string()), ("addr_empty", pa.bool_()),
])

def _normalize_batch(batch):
    out = []
    for raw_name, raw_addr in batch:
        n, a = normalize_name(raw_name), normalize_address(raw_addr)
        out.append((n.key, n.text, n.core, " ".join(n.suffix), len(n.core),
                    a.text, a.tokens, a.numbers, a.postcode, a.is_empty))
    return out

def prepare_source(tsv, split, source, out, pool, workers, limit=None):
    tmp = out.with_suffix(".tmp")      # renamed only on success: a crash never leaves a half file
    writer = pq.ParquetWriter(tmp, SCHEMA, compression="zstd")
    total, t0 = 0, time.perf_counter()
    try:
        for chunk in pd.read_csv(tsv, sep="\t", header=0, names=SOURCE_COLUMNS,
                                 chunksize=200_000, dtype=str, keep_default_na=False,
                                 na_values=[], nrows=limit):
            rows = list(zip(chunk["business_name"], chunk["business_address"]))
            size = max(1, len(rows) // workers + 1)
            parts = pool.map(_normalize_batch, [rows[i:i + size] for i in range(0, len(rows), size)])
            norm = [r for p in parts for r in p]
            writer.write_table(pa.table({
                "entity_id": pa.array(chunk["entity_id"].tolist(), pa.string()),
                "country": pa.array(chunk["country"].tolist(), pa.string()),
                "source": pa.array([source] * len(chunk), pa.int8()),
                "name_key": pa.array([r[0] for r in norm], pa.string()),
                "name_text": pa.array([r[1] for r in norm], pa.string()),
                "name_tokens": pa.array([r[2] for r in norm], pa.list_(pa.string())),
                "name_suffix": pa.array([r[3] for r in norm], pa.string()),
                "n_core": pa.array([r[4] for r in norm], pa.int16()),
                "addr_text": pa.array([r[5] for r in norm], pa.string()),
                "addr_tokens": pa.array([r[6] for r in norm], pa.list_(pa.string())),
                "addr_nums": pa.array([r[7] for r in norm], pa.list_(pa.string())),
                "postcode": pa.array([r[8] for r in norm], pa.string()),
                "addr_empty": pa.array([r[9] for r in norm], pa.bool_()),
            }, schema=SCHEMA))
            total += len(chunk)
    finally:
        writer.close()
    tmp.rename(out)
    print(f"  {split}/s{source}: {total:>10,} rows in {time.perf_counter() - t0:5.1f}s")

stamp = WORK_DIR / "norm_version.txt"
tag = f"{NORM_VERSION}:{CONFIG['prepare_limit']}"
if not stamp.exists() or stamp.read_text().strip() != tag:
    stale = [*WORK_DIR.glob("*.parquet"), *WORK_DIR.glob("*.tmp"), *WORK_DIR.glob("*.duckdb*"),
             WORK_DIR / "lgbm.txt", WORK_DIR / "calib.npz"]
    for f in stale:
        f.unlink(missing_ok=True)
    print(f"  stage 0 invalidated -> cleared work/ (now {tag})")

with mp.Pool(CONFIG["workers"]) as pool:
    for split, source, name in SOURCES:
        if split not in CONFIG["prepare_splits"]:
            continue
        out = WORK_DIR / f"{split}_s{source}.parquet"
        if out.exists():
            print(f"  {split}/s{source}: exists ({pq.ParquetFile(out).metadata.num_rows:,} rows), skip")
            continue
        prepare_source(DATA_DIR / split / name, split, source, out, pool,
                       CONFIG["workers"], CONFIG["prepare_limit"])
stamp.write_text(tag)

print("\nrow-count check / empty-name rate:")
with duckdb.connect() as c0:
    for split, src, _ in SOURCES:
        if split not in CONFIG["prepare_splits"]:
            continue
        p = WORK_DIR / f"{split}_s{src}.parquet"
        n, e = c0.sql(f"SELECT COUNT(*), SUM((n_core = 0)::INT) FROM '{p}'").fetchone()
        exp = EXPECTED_ROWS[f"{split}_s{src}"]
        ok = CONFIG["prepare_limit"] is not None or n == exp
        print(f"  {split}/s{src} {n:>11,} rows {'OK' if ok else f'MISMATCH (expected {exp:,})'}"
              f"  empty names {100 * e / n:.4f}% {'OK' if e / n <= 1e-4 else 'CHECK'}")

## The metric
F0.5 = 1.25·h / (k + 0.25·m); empty vs empty = 1; any prediction on a singleton = 0.

In [ ]:
def f05(pred: set, true: set) -> float:
    k, m = len(pred), len(true)
    if k == 0 and m == 0:
        return 1.0
    if k == 0 or m == 0:
        return 0.0
    h = len(pred & true)
    return 0.0 if h == 0 else 1.25 * h / (k + 0.25 * m)

def macro_f05(pred: dict, truth: dict) -> float:
    if not truth:
        return 0.0
    return sum(f05(pred.get(s, set()), t) for s, t in truth.items()) / len(truth)

def load_ground_truth(s1_ids=None) -> dict:
    g = pd.read_csv(DATA_DIR / "train" / GROUND_TRUTH, sep="\t",
                    dtype=str, keep_default_na=False, na_values=[])
    out = {}
    for sid, matched in zip(g["source1_entity_id"], g["matched_entity_ids"]):
        if s1_ids is not None and sid not in s1_ids:
            continue
        out[sid] = set(matched.split(",")) if matched else set()
    return out

ex = f05({"S2-00047", "S2-00193", "S3-00812"}, {"S2-00047", "S3-00812"})
assert abs(ex - 0.714) < 0.001, ex
assert f05(set(), set()) == 1.0 and f05({"x"}, set()) == 0.0 and f05({"a"}, {"a"}) == 1.0
print(f"metric checks PASS (README example = {ex:.3f})")

## Stage 1 — blocking (schemes A, A2, B–H; fair-slot union)

In [ ]:
DOMAIN_TOKENS = "('com','www','net','org','co','in','io','biz','info','html')"
SQUASH_SQL = f"array_to_string(list_filter(name_tokens, t -> t NOT IN {DOMAIN_TOKENS}), '')"
SKELETON_SQL = ("regexp_replace(regexp_replace(regexp_replace(regexp_replace("
                "tok,'sh','s','g'),'ph','f','g'),'c','k','g'),'[aeiou]','','g')")
# Scheme G: up to 3 address numbers plus the postcode (5-digit US house numbers
# get parsed as ZIP codes, so the postcode doubles as a house number).
GNUM_SQL = ("CASE WHEN postcode <> '' THEN list_append(list_slice(addr_nums, 1, 3), postcode) "
            "ELSE list_slice(addr_nums, 1, 3) END")

def connect(threads=None, mem_gb=None, spill_gb=None, db=None):
    """Hard spill ceiling, so a bad config fails instead of filling the disk."""
    threads = threads or CONFIG["threads"]; mem_gb = mem_gb or CONFIG["mem_gb"]
    spill_gb = spill_gb or CONFIG["spill_gb"]
    con = duckdb.connect(str(db) if db else ":memory:")
    tmp = WORK_DIR / "duckdb_tmp"; tmp.mkdir(parents=True, exist_ok=True)
    free_gb = shutil.disk_usage(tmp).free / 2**30
    if free_gb < spill_gb + 2:
        spill_gb = max(1, int(free_gb) - 2)
        print(f"  ! only {free_gb:.1f}GB free, capping spill at {spill_gb}GB")
    con.execute(f"SET threads={threads}")
    con.execute(f"SET memory_limit='{mem_gb}GB'")
    con.execute(f"SET temp_directory='{tmp}'")
    con.execute(f"SET max_temp_directory_size='{spill_gb}GB'")
    con.execute("SET preserve_insertion_order=false")
    return con

def estimate_pairs(con, table, label, budget=None):
    budget = budget or CONFIG["pair_budget"]
    est = int(con.sql(f"SELECT COALESCE(SUM(df),0) FROM {table}").fetchone()[0] or 0)
    print(f"  estimate {label:<10s} {est:>14,} pairs  [{'OK' if est <= budget else 'OVER BUDGET'}]")
    if est > budget:
        raise RuntimeError(f"{label}: ~{est:,} pairs exceeds budget {budget:,}.")
    return est

def _step(con, label, sql):
    t = time.perf_counter(); con.execute(sql)
    print(f"  {label:<22s} {time.perf_counter() - t:6.1f}s", flush=True)

In [ ]:
def build_pool(con, split):
    pool = [str(WORK_DIR / f"{split}_s2.parquet"), str(WORK_DIR / f"{split}_s3.parquet")]
    con.execute(f"""CREATE OR REPLACE VIEW cand AS
        SELECT entity_id, country, name_key, name_tokens, addr_tokens, postcode, addr_nums
        FROM read_parquet({pool})""")
    n_cand = con.sql("SELECT COUNT(*) FROM cand").fetchone()[0]
    print(f"  pool {n_cand:,} records")

    _step(con, "name tokens", """
        CREATE OR REPLACE TABLE cand_tok AS
        SELECT entity_id, country, UNNEST(name_tokens) AS tok FROM cand""")
    _step(con, "token df", f"""
        CREATE OR REPLACE TABLE tokdf AS
        SELECT country, tok, COUNT(*)::BIGINT AS df, LN(1.0 + {n_cand}.0 / COUNT(*)) AS idf
        FROM cand_tok GROUP BY country, tok""")
    _step(con, "squash keys", f"""
        CREATE OR REPLACE TABLE cand_keys AS
        SELECT entity_id, country, {SQUASH_SQL} AS squash FROM cand""")
    _step(con, "skeletons", f"""
        CREATE OR REPLACE TABLE cand_skel AS
        SELECT entity_id, country, skel FROM (
            SELECT entity_id, country, {SKELETON_SQL} AS skel FROM cand_tok
        ) WHERE length(skel) >= 3;
        CREATE OR REPLACE TABLE skeldf AS
        SELECT country, skel, COUNT(*)::BIGINT AS df, LN(1.0 + {n_cand}.0 / COUNT(*)) AS idf
        FROM cand_skel GROUP BY country, skel;""")
    _step(con, "address tokens", f"""
        CREATE OR REPLACE TABLE cand_atok AS
        SELECT entity_id, country, UNNEST(addr_tokens) AS atok FROM cand;
        CREATE OR REPLACE TABLE atokdf AS
        SELECT country, atok, COUNT(*)::BIGINT AS df, LN(1.0 + {n_cand}.0 / COUNT(*)) AS idf
        FROM cand_atok GROUP BY country, atok;""")
    _step(con, "token pairs", f"""
        CREATE OR REPLACE TABLE cand_rare AS
        SELECT entity_id, country, tok FROM (
            SELECT c.entity_id, c.country, c.tok,
                   ROW_NUMBER() OVER (PARTITION BY c.entity_id ORDER BY t.df ASC, c.tok) AS rk
            FROM cand_tok c JOIN tokdf t ON t.country = c.country AND t.tok = c.tok
        ) WHERE rk <= {CONFIG['pair_tokens']};
        CREATE OR REPLACE TABLE cand_pair AS
        SELECT a.entity_id, a.country, a.tok || '|' || b.tok AS pk
        FROM cand_rare a JOIN cand_rare b ON b.entity_id = a.entity_id AND b.tok > a.tok;
        CREATE OR REPLACE TABLE pairdf AS
        SELECT country, pk, COUNT(*)::BIGINT AS df, LN(1.0 + {n_cand}.0 / COUNT(*)) AS idf
        FROM cand_pair GROUP BY country, pk;""")
    _step(con, "addr num keys (G)", f"""
        CREATE OR REPLACE TABLE cand_g AS
        SELECT DISTINCT t.entity_id, t.country, n.num || '|' || t.atok AS gk
        FROM (
            SELECT entity_id, country, atok FROM (
                SELECT c.entity_id, c.country, c.atok,
                       ROW_NUMBER() OVER (PARTITION BY c.entity_id ORDER BY a.df ASC, c.atok) AS rk
                FROM cand_atok c JOIN atokdf a ON a.country = c.country AND a.atok = c.atok
                WHERE length(c.atok) >= 3
            ) WHERE rk <= {CONFIG['g_tokens']}
        ) t
        JOIN (SELECT entity_id, UNNEST({GNUM_SQL}) AS num FROM cand) n ON n.entity_id = t.entity_id;
        CREATE OR REPLACE TABLE gdf AS
        SELECT country, gk, COUNT(*)::BIGINT AS df, LN(1.0 + {n_cand}.0 / COUNT(*)) AS idf
        FROM cand_g GROUP BY country, gk;""")
    _step(con, "name x addr keys (H)", f"""
        CREATE OR REPLACE TABLE cand_h AS
        SELECT DISTINCT n.entity_id, n.country, n.tok || '|' || a.atok AS hk
        FROM (
            SELECT entity_id, country, tok FROM (
                SELECT c.entity_id, c.country, c.tok,
                       ROW_NUMBER() OVER (PARTITION BY c.entity_id ORDER BY td.df ASC, c.tok) AS rk
                FROM cand_tok c JOIN tokdf td ON td.country = c.country AND td.tok = c.tok
                WHERE length(c.tok) >= 3 AND c.tok NOT IN {DOMAIN_TOKENS}
            ) WHERE rk <= {CONFIG['h_name_tokens']}
        ) n
        JOIN (
            SELECT entity_id, atok FROM (
                SELECT c.entity_id, c.atok,
                       ROW_NUMBER() OVER (PARTITION BY c.entity_id ORDER BY ad.df ASC, c.atok) AS rk
                FROM cand_atok c JOIN atokdf ad ON ad.country = c.country AND ad.atok = c.atok
                WHERE length(c.atok) >= 3
            ) WHERE rk <= {CONFIG['h_addr_tokens']}
        ) a ON a.entity_id = n.entity_id;
        CREATE OR REPLACE TABLE hdf AS
        SELECT country, hk, COUNT(*)::BIGINT AS df, LN(1.0 + {n_cand}.0 / COUNT(*)) AS idf
        FROM cand_h GROUP BY country, hk;""")
    return n_cand

In [ ]:
def build_shard(con, split, s1_filter, suffix="", top_k=None):
    cfg, top_k = CONFIG, top_k or CONFIG["top_k"]
    s1p = str(WORK_DIR / f"{split}_s1.parquet")
    con.execute(f"""CREATE OR REPLACE VIEW s1 AS
        SELECT entity_id, country, name_key, name_tokens, addr_tokens, postcode, addr_nums
        FROM read_parquet('{s1p}') {s1_filter}""")
    n_s1 = con.sql("SELECT COUNT(*) FROM s1").fetchone()[0]
    print(f"  S1 rows {n_s1:,}")

    _step(con, "s1 rare tokens", f"""
        CREATE OR REPLACE TABLE s1_rare AS
        SELECT entity_id, country, tok, df, idf FROM (
            SELECT s.entity_id, s.country, t.tok, t.df, t.idf,
                   ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY t.df ASC, t.tok) AS rk
            FROM (SELECT entity_id, country, UNNEST(name_tokens) AS tok FROM s1) s
            JOIN tokdf t ON t.country = s.country AND t.tok = s.tok
        ) WHERE rk <= {cfg['rare_per_s1']} AND df <= {cfg['max_token_df']}""")
    estimate_pairs(con, "s1_rare", "scheme B")

    _step(con, "s1 skeletons", f"""
        CREATE OR REPLACE TABLE s1_skel AS
        SELECT entity_id, country, skel, df, idf FROM (
            SELECT s.entity_id, s.country, k.skel, k.df, k.idf,
                   ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY k.df ASC, k.skel) AS rk
            FROM (SELECT entity_id, country, {SKELETON_SQL} AS skel
                  FROM (SELECT entity_id, country, UNNEST(name_tokens) AS tok FROM s1)) s
            JOIN skeldf k ON k.country = s.country AND k.skel = s.skel
            WHERE length(s.skel) >= 3
        ) WHERE rk <= {cfg['rare_per_s1']} AND df <= {cfg['skel_max_df']}""")
    estimate_pairs(con, "s1_skel", "scheme C")

    _step(con, "s1 rare addr", f"""
        CREATE OR REPLACE TABLE s1_arare AS
        SELECT entity_id, country, atok, df, idf FROM (
            SELECT s.entity_id, s.country, t.atok, t.df, t.idf,
                   ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY t.df ASC, t.atok) AS rk
            FROM (SELECT entity_id, country, UNNEST(addr_tokens) AS atok FROM s1) s
            JOIN atokdf t ON t.country = s.country AND t.atok = s.atok
        ) WHERE rk <= {cfg['addr_rare_per_s1']} AND df <= {cfg['addr_max_df']}""")
    estimate_pairs(con, "s1_arare", "scheme E")

    _step(con, "s1 token pairs", f"""
        CREATE OR REPLACE TABLE s1_rare4 AS
        SELECT entity_id, country, tok FROM (
            SELECT s.entity_id, s.country, s.tok,
                   ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY t.df ASC, s.tok) AS rk
            FROM (SELECT entity_id, country, UNNEST(name_tokens) AS tok FROM s1) s
            JOIN tokdf t ON t.country = s.country AND t.tok = s.tok
        ) WHERE rk <= {cfg['pair_tokens']};
        CREATE OR REPLACE TABLE s1_pair AS
        SELECT entity_id, country, pk, df, idf FROM (
            SELECT sp.entity_id, sp.country, sp.pk, d.df, d.idf,
                   ROW_NUMBER() OVER (PARTITION BY sp.entity_id ORDER BY d.df ASC, sp.pk) AS rk
            FROM (SELECT a.entity_id, a.country, a.tok || '|' || b.tok AS pk
                  FROM s1_rare4 a JOIN s1_rare4 b ON b.entity_id = a.entity_id AND b.tok > a.tok) sp
            JOIN pairdf d ON d.country = sp.country AND d.pk = sp.pk
        ) WHERE rk <= {cfg['pair_per_s1']} AND df <= {cfg['pair_max_df']}""")
    estimate_pairs(con, "s1_pair", "scheme F")

    _step(con, "s1 addr num keys", f"""
        CREATE OR REPLACE TABLE s1_g AS
        SELECT entity_id, country, gk, df, idf FROM (
            SELECT k.entity_id, k.country, k.gk, d.df, d.idf,
                   ROW_NUMBER() OVER (PARTITION BY k.entity_id ORDER BY d.df ASC, k.gk) AS rk
            FROM (
                SELECT DISTINCT t.entity_id, t.country, n.num || '|' || t.atok AS gk
                FROM (
                    SELECT entity_id, country, atok FROM (
                        SELECT s.entity_id, s.country, s.atok,
                               ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY a.df ASC, s.atok) AS rk
                        FROM (SELECT entity_id, country, UNNEST(addr_tokens) AS atok FROM s1) s
                        JOIN atokdf a ON a.country = s.country AND a.atok = s.atok
                        WHERE length(s.atok) >= 3
                    ) WHERE rk <= {cfg['g_tokens']}
                ) t
                JOIN (SELECT entity_id, UNNEST({GNUM_SQL}) AS num FROM s1) n ON n.entity_id = t.entity_id
            ) k JOIN gdf d ON d.country = k.country AND d.gk = k.gk
        ) WHERE rk <= {cfg['g_per_s1']} AND df <= {cfg['g_max_df']}""")
    estimate_pairs(con, "s1_g", "scheme G")

    _step(con, "s1 name x addr keys", f"""
        CREATE OR REPLACE TABLE s1_h AS
        SELECT entity_id, country, hk, df, idf FROM (
            SELECT k.entity_id, k.country, k.hk, d.df, d.idf,
                   ROW_NUMBER() OVER (PARTITION BY k.entity_id ORDER BY d.df ASC, k.hk) AS rk
            FROM (
                SELECT DISTINCT n.entity_id, n.country, n.tok || '|' || a.atok AS hk
                FROM (
                    SELECT entity_id, country, tok FROM (
                        SELECT s.entity_id, s.country, s.tok,
                               ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY td.df ASC, s.tok) AS rk
                        FROM (SELECT entity_id, country, UNNEST(name_tokens) AS tok FROM s1) s
                        JOIN tokdf td ON td.country = s.country AND td.tok = s.tok
                        WHERE length(s.tok) >= 3 AND s.tok NOT IN {DOMAIN_TOKENS}
                    ) WHERE rk <= {cfg['h_name_tokens']}
                ) n
                JOIN (
                    SELECT entity_id, atok FROM (
                        SELECT s.entity_id, s.atok,
                               ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY ad.df ASC, s.atok) AS rk
                        FROM (SELECT entity_id, country, UNNEST(addr_tokens) AS atok FROM s1) s
                        JOIN atokdf ad ON ad.country = s.country AND ad.atok = s.atok
                        WHERE length(s.atok) >= 3
                    ) WHERE rk <= {cfg['h_addr_tokens']}
                ) a ON a.entity_id = n.entity_id
            ) k JOIN hdf d ON d.country = k.country AND d.hk = k.hk
        ) WHERE rk <= {cfg['h_per_s1']} AND df <= {cfg['h_max_df']}""")
    estimate_pairs(con, "s1_h", "scheme H")

    # ---- the schemes; each keeps its internal rank rk ----
    _step(con, "A exact key", f"""
        CREATE OR REPLACE TABLE pairs_a AS
        SELECT s1_id, cand_id, rk FROM (
            SELECT s.entity_id AS s1_id, c.entity_id AS cand_id,
                   ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY
                       (c.postcode <> '' AND c.postcode = s.postcode) DESC, c.entity_id) AS rk
            FROM s1 s JOIN cand c ON c.country = s.country AND c.name_key = s.name_key
            WHERE s.name_key <> ''
        ) WHERE rk <= {cfg['key_cap']}""")
    _step(con, "A2 squash", f"""
        CREATE OR REPLACE TABLE pairs_a2 AS
        SELECT s1_id, cand_id, rk FROM (
            SELECT s.entity_id AS s1_id, c.entity_id AS cand_id,
                   ROW_NUMBER() OVER (PARTITION BY s.entity_id ORDER BY c.entity_id) AS rk
            FROM (SELECT entity_id, country, {SQUASH_SQL} AS squash FROM s1) s
            JOIN cand_keys c ON c.country = s.country AND c.squash = s.squash
            WHERE length(s.squash) >= 4
        ) WHERE rk <= {cfg['key_cap']}""")

    def ranked(tag, key_tbl, key_col, cand_tbl, cap, having=""):
        _step(con, f"{tag} scheme", f"""
            CREATE OR REPLACE TABLE pairs_{tag} AS
            SELECT s1_id, cand_id, idf_score, shared, rk FROM (
                SELECT s1_id, cand_id, idf_score, shared,
                       ROW_NUMBER() OVER (PARTITION BY s1_id
                           ORDER BY idf_score DESC, shared DESC, cand_id) AS rk
                FROM (
                    SELECT r.entity_id AS s1_id, c.entity_id AS cand_id,
                           SUM(r.idf) AS idf_score, COUNT(*)::INT AS shared
                    FROM {key_tbl} r JOIN {cand_tbl} c
                      ON c.country = r.country AND c.{key_col} = r.{key_col}
                    GROUP BY 1, 2 {having}
                )
            ) WHERE rk <= {cap}""")

    ranked("b", "s1_rare", "tok", "cand_tok", cfg["scheme_cap"])
    ranked("c", "s1_skel", "skel", "cand_skel", cfg["scheme_cap"])
    ranked("e", "s1_arare", "atok", "cand_atok", cfg["scheme_cap"], "HAVING COUNT(*) >= 2")
    ranked("f", "s1_pair", "pk", "cand_pair", cfg["scheme_cap"])
    ranked("g", "s1_g", "gk", "cand_g", cfg["g_cap"])
    ranked("h", "s1_h", "hk", "cand_h", cfg["h_cap"])

    _step(con, "D postcode", """
        CREATE OR REPLACE TABLE pairs_d AS
        SELECT s1_id, cand_id, shared,
               ROW_NUMBER() OVER (PARTITION BY s1_id ORDER BY shared DESC, cand_id) AS rk
        FROM (
            SELECT s1_id, cand_id, COUNT(*)::INT AS shared FROM (
                SELECT s.entity_id AS s1_id, c.entity_id AS cand_id
                FROM (SELECT entity_id, country, postcode, UNNEST(name_tokens) AS tok
                      FROM s1 WHERE postcode <> '') s
                JOIN (SELECT entity_id, country, postcode, UNNEST(name_tokens) AS tok
                      FROM cand WHERE postcode <> '') c
                  ON c.country = s.country AND c.postcode = s.postcode AND c.tok = s.tok
            ) GROUP BY 1, 2
        )""")

    _step(con, "s1 idf norm", """
        CREATE OR REPLACE TABLE s1_norm AS
        SELECT s.entity_id, SUM(COALESCE(t.idf, 0.0)) AS total_idf
        FROM (SELECT entity_id, country, UNNEST(name_tokens) AS tok FROM s1) s
        LEFT JOIN tokdf t ON t.country = s.country AND t.tok = s.tok
        GROUP BY 1""")
    # Fair slots: a candidate's priority is its BEST rank inside any scheme (all #1s,
    # then all #2s, ...), so one scheme can't crowd out the others.
    _step(con, "union (uncapped)", """
        CREATE OR REPLACE TABLE upairs AS
        SELECT u.s1_id, u.cand_id, u.by_a, u.by_a2, u.by_b, u.by_c, u.by_d, u.by_e, u.by_f,
               u.by_g, u.by_h, u.idf_score, u.shared,
               u.idf_score / NULLIF(n.total_idf, 0) AS containment,
               ROW_NUMBER() OVER (PARTITION BY u.s1_id ORDER BY
                   u.best_srk ASC, GREATEST(u.by_a, u.by_a2) DESC,
                   u.idf_score / NULLIF(n.total_idf, 0) DESC,
                   u.shared DESC, u.idf_score DESC, u.cand_id) AS rnk
        FROM (
            SELECT s1_id, cand_id, MAX(by_a) by_a, MAX(by_a2) by_a2, MAX(by_b) by_b,
                   MAX(by_c) by_c, MAX(by_d) by_d, MAX(by_e) by_e, MAX(by_f) by_f,
                   MAX(by_g) by_g, MAX(by_h) by_h,
                   MAX(idf_score) idf_score, MAX(shared) shared, MIN(srk) best_srk
            FROM (
                SELECT s1_id, cand_id, 1 by_a, 0 by_a2, 0 by_b, 0 by_c, 0 by_d, 0 by_e, 0 by_f,
                       0 by_g, 0 by_h, 0.0::DOUBLE idf_score, 0 shared, rk srk FROM pairs_a
                UNION ALL SELECT s1_id,cand_id,0,1,0,0,0,0,0,0,0,0.0,0,rk FROM pairs_a2
                UNION ALL SELECT s1_id,cand_id,0,0,1,0,0,0,0,0,0,idf_score,shared,rk FROM pairs_b
                UNION ALL SELECT s1_id,cand_id,0,0,0,1,0,0,0,0,0,idf_score,shared,rk FROM pairs_c
                UNION ALL SELECT s1_id,cand_id,0,0,0,0,1,0,0,0,0,0.0,shared,rk FROM pairs_d
                UNION ALL SELECT s1_id,cand_id,0,0,0,0,0,1,0,0,0,idf_score,shared,rk FROM pairs_e
                UNION ALL SELECT s1_id,cand_id,0,0,0,0,0,0,1,0,0,idf_score,shared,rk FROM pairs_f
                UNION ALL SELECT s1_id,cand_id,0,0,0,0,0,0,0,1,0,idf_score,shared,rk FROM pairs_g
                UNION ALL SELECT s1_id,cand_id,0,0,0,0,0,0,0,0,1,idf_score,shared,rk FROM pairs_h
            ) GROUP BY s1_id, cand_id
        ) u LEFT JOIN s1_norm n ON n.entity_id = u.s1_id""")
    _step(con, "top-k cut", f"CREATE OR REPLACE TABLE candidates AS SELECT * FROM upairs WHERE rnk <= {top_k}")

    out = WORK_DIR / f"{split}_candidates{suffix}.parquet"
    con.execute(f"COPY candidates TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    for t in ("pairs_a", "pairs_a2", "pairs_b", "pairs_c", "pairs_d", "pairs_e",
              "pairs_f", "pairs_g", "pairs_h", "upairs", "candidates"):
        print(f"  {t:<12s} {con.sql(f'SELECT COUNT(*) FROM {t}').fetchone()[0]:>13,}")
    return out

### Run blocking on the training sample
File-backed DuckDB: the pool index with G/H keys does not fit 16 GB in memory.

In [ ]:
DEV_DB = WORK_DIR / "er_train.duckdb"
for suf in ("", ".wal"):
    Path(f"{DEV_DB}{suf}").unlink(missing_ok=True)
con = connect(db=DEV_DB)
t0 = time.perf_counter()
print("--- pool index (built once) ---")
build_pool(con, "train")

S, STEP = CONFIG["s1_sample"], 20
parts = []
con.execute("CREATE OR REPLACE TABLE upairs_all (s1_id VARCHAR, cand_id VARCHAR, rnk BIGINT)")
for lo in range(0, S, STEP):
    hi = min(lo + STEP, S)
    print(f"\n--- slice: S1 hash {lo}-{hi} per mille ---")
    parts.append(str(build_shard(con, "train",
        f"WHERE hash(entity_id) % 1000 >= {lo} AND hash(entity_id) % 1000 < {hi}",
        suffix=f"_p{lo:03d}")))
    con.execute("INSERT INTO upairs_all SELECT s1_id, cand_id, rnk FROM upairs")
cand_path = WORK_DIR / "train_candidates.parquet"
con.execute(f"COPY (SELECT * FROM read_parquet({parts})) TO '{cand_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")
for p in parts:
    Path(p).unlink()

# diagnostics below must see the whole sample, not only the last slice
con.execute(f"""CREATE OR REPLACE VIEW s1 AS SELECT * FROM read_parquet('{WORK_DIR / "train_s1.parquet"}')
                WHERE hash(entity_id) % 1000 < {S}""")
con.execute("CREATE OR REPLACE TABLE upairs AS SELECT * FROM upairs_all")
con.execute(f"CREATE OR REPLACE TABLE candidates AS SELECT * FROM read_parquet('{cand_path}')")
print(f"\ntotal {time.perf_counter() - t0:.1f}s -> {cand_path.name}")

In [ ]:
def diagnose(con, ks=(10, 30, 50, 100, 200, 500)):
    gt_path = str(DATA_DIR / "train" / GROUND_TRUTH)
    con.execute(f"""
        CREATE OR REPLACE TABLE gt AS
        SELECT g.s1_id, TRIM(g.cand_id) AS cand_id FROM (
            SELECT source1_entity_id AS s1_id,
                   UNNEST(STRING_SPLIT(matched_entity_ids, ',')) AS cand_id
            FROM read_csv('{gt_path}', delim='\t', header=true,
                 columns={{'source1_entity_id':'VARCHAR','matched_entity_ids':'VARCHAR'}})
            WHERE matched_entity_ids IS NOT NULL AND matched_entity_ids <> ''
        ) g JOIN s1 ON s1.entity_id = g.s1_id""")
    cols = ", ".join(f"SUM(CASE WHEN rnk<={k} THEN 1 ELSE 0 END)*1.0/COUNT(*) AS r{k}" for k in ks)
    row = con.sql(f"""
        WITH m AS (SELECT g.s1_id, g.cand_id, u.rnk FROM gt g
                   LEFT JOIN upairs u ON u.s1_id=g.s1_id AND u.cand_id=g.cand_id)
        SELECT COUNT(*), SUM(CASE WHEN rnk IS NOT NULL THEN 1 ELSE 0 END)*1.0/COUNT(*), {cols}
        FROM m""").fetchone()
    print(f"  true pairs            {row[0]:,}")
    print(f"  uncapped (generation) {row[1]:.4f}")
    for k, v in zip(ks, row[2:]):
        print(f"  recall@{k:<4d}          {v:.4f}")

def blocking_report(candidates: dict, truth: dict):
    n = len(truth); tp = cp = hits = 0
    rec = perfect = zero = ceil_sum = 0.0
    n_single = 0
    for sid, tset in truth.items():
        cand = candidates.get(sid, set()); hit = tset & cand
        cp += len(cand); tp += len(tset); hits += len(hit)
        if tset:
            r = len(hit) / len(tset); rec += r; perfect += (r == 1.0); zero += (r == 0.0)
        else:
            n_single += 1; rec += 1.0; perfect += 1.0
        ceil_sum += f05(hit, tset)
    print(f"  entities              {n:,}")
    print(f"  mean candidates/ent   {cp / n:.1f}")
    print(f"  recall (micro)        {hits / tp:.4f}")
    print(f"  entities zero recall  {zero / max(1, n - n_single):.4f}")
    print(f"  CEILING macro F_0.5   {ceil_sum / n:.4f}")

print("=== RECALL@K ===");  diagnose(con)
rows = con.sql("SELECT s1_id, cand_id FROM candidates").fetchnumpy()
cands = {}
for s, c in zip(rows["s1_id"], rows["cand_id"]):
    cands.setdefault(s, set()).add(c)
s1_ids = set(con.sql("SELECT entity_id FROM s1").fetchnumpy()["entity_id"])
print("\n=== BLOCKING REPORT ===");  blocking_report(cands, load_ground_truth(s1_ids))

In [ ]:
def inspect_misses(con, n=15):
    pool = [str(WORK_DIR / "train_s2.parquet"), str(WORK_DIR / "train_s3.parquet")]
    s1p = str(WORK_DIR / "train_s1.parquet")
    rows = con.sql(f"""
        WITH miss AS (SELECT g.s1_id, g.cand_id FROM gt g
                      LEFT JOIN upairs u ON u.s1_id=g.s1_id AND u.cand_id=g.cand_id
                      WHERE u.s1_id IS NULL)
        SELECT s.name_text, p.name_text, s.name_tokens, p.name_tokens, s.country,
               s.addr_tokens, p.addr_tokens
        FROM miss m JOIN read_parquet('{s1p}') s ON s.entity_id=m.s1_id
        JOIN read_parquet({pool}) p ON p.entity_id=m.cand_id
        USING SAMPLE {n} ROWS""").fetchall()
    for sn, cn, st, ct, country, sat, cat in rows:
        ov = set(st or ()) & set(ct or ()); aov = set(sat or ()) & set(cat or ())
        print(f"  [{country:6s}] {sn!r:36s} vs {cn!r:36s} shared={sorted(ov)} addr={sorted(aov)[:4]}")

inspect_misses(con)
con.close()
for suf in ("", ".wal"):
    Path(f"{DEV_DB}{suf}").unlink(missing_ok=True)   # free the disk for the test run

## Stage 2 — pair features
`pair_features` below is the original row-by-row version. It is kept only as the
reference that `check_features` compares the fast version against.

In [ ]:
FEATURE_COLUMNS = [
    "n_token_set", "n_token_sort", "n_partial", "n_jaro", "n_prefix", "n_jacc",
    "n_idf_cover_s1", "n_idf_cover_c", "n_len_ratio", "n_tok_diff", "n_exact_key",
    "n_suffix_agree", "n_suffix_conflict",
    "n_skel_jacc", "n_skel_ratio", "n_extra_cnt", "n_extra_max_idf",
    "n_missing_cnt", "n_missing_max_idf",
    "a_token_set", "a_jacc", "a_post_match", "a_post_both", "a_num_overlap",
    "a_num_both", "a_empty_either",
    "b_by_a", "b_by_a2", "b_by_b", "b_by_c", "b_by_d", "b_by_e", "b_by_f", "b_by_g", "b_by_h",
    "b_n_schemes", "b_idf_score", "b_shared", "b_containment", "b_rank",
    "g_n_cands", "g_rank_frac", "g_score_ratio", "g_score_margin", "g_is_top",
    "c_source", "c_same_country",
    "x_s1_name_dup", "x_c_name_s1cnt", "x_rival_addr", "x_rival_addr_adv", "x_rival_num",
]
IDX = {c: i for i, c in enumerate(FEATURE_COLUMNS)}
N_FEATURES = len(FEATURE_COLUMNS)

# Sound-alike skeleton ('kansaltansi' ~ 'consultancy').
_SKEL_RULES = [(re.compile(p), r) for p, r in [
    (r"ph", "f"), (r"sh", "s"), (r"ck", "k"), (r"c(?=[eiy])", "s"), (r"c", "k"),
    (r"q", "k"), (r"x", "ks"), (r"z", "s"), (r"w", "v"), (r"j", "g"), (r"h", ""),
    (r"[aeiouy]", ""), (r"(.)\1+", r"\1")]]

@lru_cache(maxsize=2_000_000)
def skel(tok):
    for pat, rep in _SKEL_RULES:
        tok = pat.sub(rep, tok)
    return tok

_UNSEEN_IDF = 16.0

def _leftovers(a, b, b_sk):
    return [t for t in a if t not in b and not (len(skel(t)) >= 2 and skel(t) in b_sk)]

def _jacc(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

def pair_features(df, idf):
    out = np.zeros((len(df), N_FEATURES), dtype=np.float32); I = IDX
    for i, r in enumerate(df.itertuples(index=False)):
        sn, cn = r.s_name_text or "", r.c_name_text or ""
        st = set(r.s_name_tokens) if r.s_name_tokens is not None else set()
        ct = set(r.c_name_tokens) if r.c_name_tokens is not None else set()
        country = r.s_country
        out[i, I["n_token_set"]] = fuzz.token_set_ratio(sn, cn) / 100.0
        out[i, I["n_token_sort"]] = fuzz.token_sort_ratio(sn, cn) / 100.0
        out[i, I["n_partial"]] = fuzz.partial_ratio(sn, cn) / 100.0
        out[i, I["n_jaro"]] = JaroWinkler.similarity(sn, cn)
        out[i, I["n_prefix"]] = fuzz.QRatio(sn[:12], cn[:12]) / 100.0
        out[i, I["n_jacc"]] = _jacc(st, ct)
        ws = sum(idf.get((country, t), 0.0) for t in st & ct)
        w_s = sum(idf.get((country, t), 0.0) for t in st)
        w_c = sum(idf.get((country, t), 0.0) for t in ct)
        out[i, I["n_idf_cover_s1"]] = ws / w_s if w_s else 0.0
        out[i, I["n_idf_cover_c"]] = ws / w_c if w_c else 0.0
        ls, lc = len(sn), len(cn)
        out[i, I["n_len_ratio"]] = min(ls, lc) / max(ls, lc) if max(ls, lc) else 0.0
        out[i, I["n_tok_diff"]] = abs(len(st) - len(ct))
        out[i, I["n_exact_key"]] = float(r.s_name_key == r.c_name_key and r.s_name_key != "")
        ssuf, csuf = set((r.s_name_suffix or "").split()), set((r.c_name_suffix or "").split())
        out[i, I["n_suffix_agree"]] = float(bool(ssuf & csuf))
        out[i, I["n_suffix_conflict"]] = float(bool(ssuf) and bool(csuf) and not (ssuf & csuf))
        s_sk = {k for k in (skel(t) for t in st) if len(k) >= 2}
        c_sk = {k for k in (skel(t) for t in ct) if len(k) >= 2}
        out[i, I["n_skel_jacc"]] = _jacc(s_sk, c_sk)
        out[i, I["n_skel_ratio"]] = fuzz.ratio(" ".join(sorted(s_sk)), " ".join(sorted(c_sk))) / 100.0
        extra = _leftovers(ct, st, s_sk)
        missing = _leftovers(st, ct, c_sk)
        out[i, I["n_extra_cnt"]] = len(extra)
        out[i, I["n_extra_max_idf"]] = max((idf.get((country, t), _UNSEEN_IDF) for t in extra), default=0.0)
        out[i, I["n_missing_cnt"]] = len(missing)
        out[i, I["n_missing_max_idf"]] = max((idf.get((country, t), _UNSEEN_IDF) for t in missing), default=0.0)
        sa, ca = r.s_addr_text or "", r.c_addr_text or ""
        sat = set(r.s_addr_tokens) if r.s_addr_tokens is not None else set()
        cat = set(r.c_addr_tokens) if r.c_addr_tokens is not None else set()
        out[i, I["a_token_set"]] = fuzz.token_set_ratio(sa, ca) / 100.0 if sa and ca else 0.0
        out[i, I["a_jacc"]] = _jacc(sat, cat)
        sp, cp_ = r.s_postcode or "", r.c_postcode or ""
        out[i, I["a_post_match"]] = float(sp != "" and sp == cp_)
        out[i, I["a_post_both"]] = float(sp != "" and cp_ != "")
        snum = set(r.s_addr_nums) if r.s_addr_nums is not None else set()
        cnum = set(r.c_addr_nums) if r.c_addr_nums is not None else set()
        out[i, I["a_num_overlap"]] = _jacc(snum, cnum)
        out[i, I["a_num_both"]] = float(bool(snum) and bool(cnum))
        out[i, I["a_empty_either"]] = float(r.s_addr_empty or r.c_addr_empty)
        a, a2, b, c, d, e, f = r.by_a, r.by_a2, r.by_b, r.by_c, r.by_d, r.by_e, r.by_f
        for nm, v in (("b_by_a", a), ("b_by_a2", a2), ("b_by_b", b), ("b_by_c", c),
                      ("b_by_d", d), ("b_by_e", e), ("b_by_f", f)):
            out[i, I[nm]] = v
        out[i, I["b_n_schemes"]] = a + a2 + b + c + d + e + f
        out[i, I["b_idf_score"]] = r.idf_score or 0.0
        out[i, I["b_shared"]] = r.shared or 0
        out[i, I["b_containment"]] = r.containment or 0.0
        out[i, I["b_rank"]] = r.rnk
        out[i, I["c_source"]] = 2.0 if str(r.cand_id).startswith("S2-") else 3.0
        out[i, I["c_same_country"]] = float(r.s_country == r.c_country)
        out[i, I["x_s1_name_dup"]] = r.x_s1_dup
        out[i, I["x_c_name_s1cnt"]] = r.x_c_s1cnt
        rj = r.x_rival_jacc
        if pd.isna(rj):
            out[i, I["x_rival_addr"]] = np.nan
            out[i, I["x_rival_addr_adv"]] = np.nan
            out[i, I["x_rival_num"]] = np.nan
        else:
            out[i, I["x_rival_addr"]] = rj
            out[i, I["x_rival_addr_adv"]] = rj - out[i, I["a_jacc"]]
            out[i, I["x_rival_num"]] = r.x_rival_num
    return out

def add_group_features(feats, s1_ids):
    """Rank/margin within each entity's own candidate list, in place."""
    base = feats[:, IDX["n_token_set"]] * 0.6 + feats[:, IDX["n_idf_cover_s1"]] * 0.4
    g = pd.DataFrame({"sid": s1_ids, "score": base}).groupby("sid")["score"]
    n = g.transform("size").to_numpy(np.float32)
    mx = g.transform("max").to_numpy(np.float32)
    rank = g.rank(ascending=False, method="first").to_numpy(np.float32)
    feats[:, IDX["g_n_cands"]] = n
    feats[:, IDX["g_rank_frac"]] = rank / np.maximum(n, 1)
    feats[:, IDX["g_score_ratio"]] = np.where(mx > 0, base / mx, 0.0)
    feats[:, IDX["g_score_margin"]] = base - mx
    feats[:, IDX["g_is_top"]] = (rank == 1).astype(np.float32)

### Fast features
Set features and the cross-S1 rival features run in DuckDB (multi-threaded); string
scores run through rapidfuzz `cpdist` on whole columns. `check_features` recomputes a
sample with the slow reference above and raises if any feature differs.

In [ ]:
GROUP_FEATURES = ["g_n_cands", "g_rank_frac", "g_score_ratio", "g_score_margin", "g_is_top"]
STRING_FEATURES = ["n_token_set", "n_token_sort", "n_partial", "n_jaro", "n_prefix",
                   "n_skel_ratio", "a_token_set"]
NAN_FEATURES = ["x_rival_addr", "x_rival_addr_adv", "x_rival_num"]   # NaN = no rival S1
SQL_FEATURES = [c for c in FEATURE_COLUMNS if c not in STRING_FEATURES + GROUP_FEATURES]

def _pool_files(split):
    return [str(WORK_DIR / f"{split}_s2.parquet"), str(WORK_DIR / f"{split}_s3.parquet")]

def load_idf(con, split):
    path = WORK_DIR / f"{split}_idf.parquet"
    if not path.exists():
        pool = _pool_files(split)
        con.execute(f"""COPY (
            WITH ct AS (SELECT country, UNNEST(name_tokens) AS tok FROM read_parquet({pool}))
            SELECT country, tok,
                   LN(1.0 + (SELECT COUNT(*) FROM read_parquet({pool})) / COUNT(*)) AS idf
            FROM ct GROUP BY country, tok
        ) TO '{path}' (FORMAT PARQUET)""")
    return path

def load_skel(con, split):
    """Sound-alike skeleton for every name token of the split, computed once."""
    path = WORK_DIR / f"{split}_skel.parquet"
    if not path.exists():
        files = [str(WORK_DIR / f"{split}_s{i}.parquet") for i in (1, 2, 3)]
        toks = con.sql(f"SELECT DISTINCT tok FROM (SELECT UNNEST(name_tokens) AS tok "
                       f"FROM read_parquet({files}))").fetchnumpy()["tok"].tolist()
        pq.write_table(pa.table({"tok": toks, "sk": [skel(t) for t in toks]}), path)
    return path

def _J(a, b):
    return (f"CASE WHEN len({a}) = 0 OR len({b}) = 0 THEN 0.0 ELSE "
            f"len(list_intersect({a}, {b}))::DOUBLE / len(list_distinct(list_concat({a}, {b}))) END")

def _leftover_sql(name, id_col, toks, other, other_sk_tbl, other_id):
    # words of one side with no exact or sound-alike counterpart on the other
    return f"""
    CREATE OR REPLACE TABLE {name} AS
    SELECT x.s1_id, x.cand_id, COUNT(*) AS n, MAX(coalesce(t.idf, {_UNSEEN_IDF})) AS mx
    FROM (SELECT b.s1_id, b.cand_id, b.s_country AS country, b.{other} AS other, o.sk AS osk,
                 UNNEST(b.{toks}) AS tok
          FROM base b LEFT JOIN {other_sk_tbl} o ON o.id = b.{other_id}) x
    LEFT JOIN skmap m ON m.tok = x.tok
    LEFT JOIN tokidf t ON t.country = x.country AND t.tok = x.tok
    WHERE NOT list_contains(x.other, x.tok)
      AND NOT coalesce(length(m.sk) >= 2 AND list_contains(x.osk, m.sk), false)
    GROUP BY x.s1_id, x.cand_id;"""

def _sk_sql(name, id_col, toks):
    return f"""
    CREATE OR REPLACE TABLE {name} AS
    SELECT x.id, list_distinct(list(m.sk)) AS sk
    FROM (SELECT DISTINCT id, tok FROM (
            SELECT id, UNNEST({toks}) AS tok FROM (SELECT DISTINCT {id_col} AS id, {toks} FROM base))) x
    JOIN skmap m ON m.tok = x.tok
    WHERE length(m.sk) >= 2
    GROUP BY x.id;"""

def _feature_setup(con, split, suffix):
    """Tables shared by all parts of one build."""
    cand = WORK_DIR / f"{split}_candidates{suffix}.parquet"
    s1p = WORK_DIR / f"{split}_s1.parquet"
    pool = _pool_files(split)
    idf_path, skel_path = load_idf(con, split), load_skel(con, split)
    con.execute(f"""
    CREATE OR REPLACE TABLE tokidf AS SELECT country, tok, idf FROM read_parquet('{idf_path}');
    CREATE OR REPLACE TABLE skmap AS SELECT tok, sk FROM read_parquet('{skel_path}');
    CREATE OR REPLACE TABLE candall AS
    SELECT s1_id, cand_id, by_a, by_a2, by_b, by_c, by_d, by_e, by_f, by_g, by_h,
           idf_score, shared, containment, rnk
    FROM read_parquet('{cand}');
    CREATE OR REPLACE TABLE ssub AS
    SELECT * FROM read_parquet('{s1p}') WHERE entity_id IN (SELECT DISTINCT s1_id FROM candall);
    CREATE OR REPLACE TABLE psub AS
    SELECT * FROM read_parquet({pool}) WHERE entity_id IN (SELECT DISTINCT cand_id FROM candall);
    CREATE OR REPLACE TABLE s1all AS
    SELECT entity_id, country, name_key, list_distinct(addr_tokens) AS at, list_distinct(addr_nums) AS an
    FROM read_parquet('{s1p}') WHERE name_key <> '';
    CREATE OR REPLACE TABLE s1cnt AS
    SELECT country, name_key, COUNT(*)::INT AS n FROM s1all GROUP BY 1, 2;
    """)
    return con.sql("SELECT COUNT(*) FROM candall").fetchone()[0]

def _feature_query(con, part, n_parts):
    con.execute(f"""
    CREATE OR REPLACE TABLE base AS
    SELECT c.s1_id, c.cand_id, c.by_a, c.by_a2, c.by_b, c.by_c, c.by_d, c.by_e, c.by_f, c.by_g, c.by_h,
           c.idf_score, c.shared, c.containment, c.rnk,
           s.country AS s_country, p.country AS c_country,
           coalesce(s.name_key, '') AS s_key, coalesce(p.name_key, '') AS c_key,
           coalesce(s.name_text, '') AS s_name, coalesce(p.name_text, '') AS c_name,
           coalesce(s.addr_text, '') AS s_addr, coalesce(p.addr_text, '') AS c_addr,
           list_distinct(s.name_tokens) AS st, list_distinct(p.name_tokens) AS ct,
           list_distinct(s.addr_tokens) AS sat, list_distinct(p.addr_tokens) AS cat,
           list_distinct(s.addr_nums) AS snum, list_distinct(p.addr_nums) AS cnum,
           list_filter(string_split(coalesce(s.name_suffix, ''), ' '), x -> x <> '') AS ssuf,
           list_filter(string_split(coalesce(p.name_suffix, ''), ' '), x -> x <> '') AS csuf,
           coalesce(s.postcode, '') AS s_post, coalesce(p.postcode, '') AS c_post,
           (s.addr_empty OR p.addr_empty) AS addr_empty_either
    FROM (SELECT * FROM candall WHERE hash(s1_id) % {n_parts} = {part}) c
    JOIN ssub s ON s.entity_id = c.s1_id
    JOIN psub p ON p.entity_id = c.cand_id;

    CREATE OR REPLACE TABLE w_s AS
    SELECT x.s1_id, SUM(t.idf) AS w
    FROM (SELECT s1_id, country, UNNEST(st) AS tok
          FROM (SELECT DISTINCT s1_id, s_country AS country, st FROM base)) x
    JOIN tokidf t ON t.country = x.country AND t.tok = x.tok GROUP BY x.s1_id;

    CREATE OR REPLACE TABLE w_c AS
    SELECT x.cand_id, SUM(t.idf) AS w
    FROM (SELECT cand_id, country, UNNEST(ct) AS tok
          FROM (SELECT DISTINCT cand_id, s_country AS country, ct FROM base)) x
    JOIN tokidf t ON t.country = x.country AND t.tok = x.tok GROUP BY x.cand_id;

    CREATE OR REPLACE TABLE w_pair AS
    SELECT x.s1_id, x.cand_id, SUM(t.idf) AS w
    FROM (SELECT s1_id, cand_id, s_country AS country, UNNEST(list_intersect(st, ct)) AS tok FROM base) x
    JOIN tokidf t ON t.country = x.country AND t.tok = x.tok GROUP BY x.s1_id, x.cand_id;

    {_sk_sql('s_sk', 's1_id', 'st')}
    {_sk_sql('c_sk', 'cand_id', 'ct')}
    {_leftover_sql('extra', 'cand_id', 'ct', 'st', 's_sk', 's1_id')}
    {_leftover_sql('missing', 's1_id', 'st', 'ct', 'c_sk', 'cand_id')}

    -- Rival fit, once per candidate (not per pair): the best and second-best address fit
    -- among S1 businesses with exactly the candidate's name. For a pair, the rival is the
    -- best one unless that is the pair's own S1, then the second best.
    CREATE OR REPLACE TABLE rtop AS
    SELECT cand_id,
           MAX(CASE WHEN rk = 1 THEN entity_id END) AS e1,
           MAX(CASE WHEN rk = 1 THEN jac END) AS j1,
           MAX(CASE WHEN rk = 2 THEN jac END) AS j2,
           MAX(cnt) AS cnt, MAX(nnum) AS nnum
    FROM (
        SELECT cand_id, entity_id, jac,
               ROW_NUMBER() OVER (PARTITION BY cand_id ORDER BY jac DESC, entity_id) AS rk,
               COUNT(*) OVER (PARTITION BY cand_id) AS cnt,
               SUM(num) OVER (PARTITION BY cand_id) AS nnum
        FROM (
            SELECT c.cand_id, r.entity_id,
                   COALESCE(len(list_intersect(r.at, c.cat))::DOUBLE
                       / NULLIF(len(list_distinct(list_concat(r.at, c.cat))), 0), 0.0) AS jac,
                   CASE WHEN len(list_intersect(r.an, c.cnum)) > 0 THEN 1 ELSE 0 END AS num
            FROM (SELECT DISTINCT cand_id, c_country, c_key, cat, cnum FROM base WHERE c_key <> '') c
            JOIN s1cnt k ON k.country = c.c_country AND k.name_key = c.c_key AND k.n <= 50
            JOIN s1all r ON r.country = c.c_country AND r.name_key = c.c_key
        )
    ) WHERE rk <= 2
    GROUP BY cand_id;

    CREATE OR REPLACE TABLE rival AS
    SELECT s1_id, cand_id,
           CASE WHEN e1 = s1_id THEN j2 ELSE j1 END AS rival_jacc,
           (nnum - (self_in AND self_num)::INT > 0)::INT AS rival_num
    FROM (
        SELECT b.s1_id, b.cand_id, t.e1, t.j1, t.j2, t.cnt, t.nnum,
               (b.s_key = b.c_key AND b.s_country = b.c_country) AS self_in,
               (len(list_intersect(b.snum, b.cnum)) > 0) AS self_num
        FROM base b JOIN rtop t ON t.cand_id = b.cand_id
    ) WHERE cnt - self_in::INT > 0;
    """)
    return f"""
    SELECT b.s1_id, b.cand_id, b.s_name, b.c_name, b.s_addr, b.c_addr,
      coalesce(array_to_string(list_sort(ss.sk), ' '), '') AS s_skstr,
      coalesce(array_to_string(list_sort(cs.sk), ' '), '') AS c_skstr,
      {_J('st', 'ct')} AS n_jacc,
      coalesce(coalesce(wp.w, 0) / NULLIF(ws.w, 0), 0) AS n_idf_cover_s1,
      coalesce(coalesce(wp.w, 0) / NULLIF(wc.w, 0), 0) AS n_idf_cover_c,
      CASE WHEN greatest(length(s_name), length(c_name)) = 0 THEN 0.0
           ELSE least(length(s_name), length(c_name))::DOUBLE
                / greatest(length(s_name), length(c_name)) END AS n_len_ratio,
      abs(len(st) - len(ct)) AS n_tok_diff,
      (s_key = c_key AND s_key <> '')::INT AS n_exact_key,
      (len(list_intersect(ssuf, csuf)) > 0)::INT AS n_suffix_agree,
      (len(ssuf) > 0 AND len(csuf) > 0 AND len(list_intersect(ssuf, csuf)) = 0)::INT AS n_suffix_conflict,
      {_J('ss.sk', 'cs.sk')} AS n_skel_jacc,
      coalesce(ex.n, 0) AS n_extra_cnt, coalesce(ex.mx, 0) AS n_extra_max_idf,
      coalesce(mi.n, 0) AS n_missing_cnt, coalesce(mi.mx, 0) AS n_missing_max_idf,
      {_J('sat', 'cat')} AS a_jacc,
      (s_post <> '' AND s_post = c_post)::INT AS a_post_match,
      (s_post <> '' AND c_post <> '')::INT AS a_post_both,
      {_J('snum', 'cnum')} AS a_num_overlap,
      (len(snum) > 0 AND len(cnum) > 0)::INT AS a_num_both,
      addr_empty_either::INT AS a_empty_either,
      by_a AS b_by_a, by_a2 AS b_by_a2, by_b AS b_by_b, by_c AS b_by_c, by_d AS b_by_d,
      by_e AS b_by_e, by_f AS b_by_f, by_g AS b_by_g, by_h AS b_by_h,
      by_a + by_a2 + by_b + by_c + by_d + by_e + by_f + by_g + by_h AS b_n_schemes,
      coalesce(idf_score, 0) AS b_idf_score, coalesce(shared, 0) AS b_shared,
      coalesce(containment, 0) AS b_containment, rnk AS b_rank,
      CASE WHEN starts_with(b.cand_id, 'S2-') THEN 2 ELSE 3 END AS c_source,
      (s_country = c_country)::INT AS c_same_country,
      coalesce(ks.n, 1) AS x_s1_name_dup, coalesce(kc.n, 0) AS x_c_name_s1cnt,
      rv.rival_jacc AS x_rival_addr,
      rv.rival_jacc - ({_J('sat', 'cat')}) AS x_rival_addr_adv,
      rv.rival_num::DOUBLE AS x_rival_num
    FROM base b
    LEFT JOIN w_pair wp ON wp.s1_id = b.s1_id AND wp.cand_id = b.cand_id
    LEFT JOIN w_s ws ON ws.s1_id = b.s1_id
    LEFT JOIN w_c wc ON wc.cand_id = b.cand_id
    LEFT JOIN s_sk ss ON ss.id = b.s1_id
    LEFT JOIN c_sk cs ON cs.id = b.cand_id
    LEFT JOIN extra ex ON ex.s1_id = b.s1_id AND ex.cand_id = b.cand_id
    LEFT JOIN missing mi ON mi.s1_id = b.s1_id AND mi.cand_id = b.cand_id
    LEFT JOIN s1cnt ks ON ks.country = b.s_country AND ks.name_key = b.s_key
    LEFT JOIN s1cnt kc ON kc.country = b.c_country AND kc.name_key = b.c_key
    LEFT JOIN rival rv ON rv.s1_id = b.s1_id AND rv.cand_id = b.cand_id
    ORDER BY b.s1_id, b.cand_id"""

def _cp(a, b, scorer, scale=100.0):
    return process.cpdist(a, b, scorer=scorer, workers=-1).astype(np.float32) / scale

def _string_features(df, feats):
    I = IDX
    sn, cn = df["s_name"].tolist(), df["c_name"].tolist()
    sa, ca = df["s_addr"].tolist(), df["c_addr"].tolist()
    feats[:, I["n_token_set"]] = _cp(sn, cn, fuzz.token_set_ratio)
    feats[:, I["n_token_sort"]] = _cp(sn, cn, fuzz.token_sort_ratio)
    feats[:, I["n_partial"]] = _cp(sn, cn, fuzz.partial_ratio)
    feats[:, I["n_jaro"]] = _cp(sn, cn, JaroWinkler.normalized_similarity, 1.0)
    feats[:, I["n_prefix"]] = _cp([s[:12] for s in sn], [c[:12] for c in cn], fuzz.QRatio)
    feats[:, I["n_skel_ratio"]] = _cp(df["s_skstr"].tolist(), df["c_skstr"].tolist(), fuzz.ratio)
    both = (df["s_addr"].to_numpy() != "") & (df["c_addr"].to_numpy() != "")
    feats[:, I["a_token_set"]] = np.where(both, _cp(sa, ca, fuzz.token_set_ratio), 0.0)

def build_features(split, suffix="", chunk=500_000, part_pairs=1_500_000):
    con = connect(mem_gb=8)           # the test run keeps a second DuckDB open
    t0 = time.perf_counter()
    n_pairs = _feature_setup(con, split, suffix)
    n_parts = max(1, math.ceil(n_pairs / part_pairs))
    print(f"  setup {time.perf_counter() - t0:.1f}s  {n_pairs:,} pairs in {n_parts} part(s)")

    out = WORK_DIR / f"{split}_features{suffix}.parquet"
    schema = pa.schema([("s1_id", pa.string()), ("cand_id", pa.string())]
                       + [(c, pa.float32()) for c in FEATURE_COLUMNS])
    writer = pq.ParquetWriter(out, schema, compression="zstd")
    sql_idx = [IDX[c] for c in SQL_FEATURES]
    fill = np.array([c not in NAN_FEATURES for c in SQL_FEATURES])
    total = 0

    def emit(df):
        nonlocal total
        feats = np.zeros((len(df), N_FEATURES), np.float32)
        vals = df[SQL_FEATURES].to_numpy(np.float32)
        vals[:, fill] = np.nan_to_num(vals[:, fill])
        feats[:, sql_idx] = vals
        _string_features(df, feats)
        add_group_features(feats, df["s1_id"].to_numpy())
        writer.write_table(pa.table(
            {"s1_id": pa.array(df["s1_id"]), "cand_id": pa.array(df["cand_id"]),
             **{c: pa.array(feats[:, j], pa.float32()) for j, c in enumerate(FEATURE_COLUMNS)}},
            schema=schema))
        total += len(df)
        print(f"    {total:>12,} pairs ({total / (time.perf_counter() - t0):,.0f}/s)", end="\r", flush=True)

    try:
        for part in range(n_parts):       # a part holds whole entities, so group features stay exact
            query = _feature_query(con, part, n_parts)
            carry = None
            for batch in con.execute(query).fetch_record_batch(chunk):
                df = batch.to_pandas()
                if carry is not None:
                    df = pd.concat([carry, df], ignore_index=True)
                tail = df["s1_id"].to_numpy() == df["s1_id"].iloc[-1]
                carry, df = df[tail], df[~tail]
                if len(df):
                    emit(df.reset_index(drop=True))
            if carry is not None and len(carry):
                emit(carry.reset_index(drop=True))
    finally:
        writer.close()
    print(f"\n  {total:,} pairs in {time.perf_counter() - t0:.1f}s -> {out.name}")
    con.close()
    if total != n_pairs:
        raise RuntimeError(f"feature rows {total:,} != candidate pairs {n_pairs:,}")
    return out

def check_features(split="train", n_ent=200):
    """Recompute a sample of entities with the slow reference; raise on any difference."""
    fast = pd.read_parquet(WORK_DIR / f"{split}_features.parquet")
    ent = fast["s1_id"].drop_duplicates()
    ids = ent.sample(min(n_ent, len(ent)), random_state=0).tolist()
    cand, s1p, pool = (WORK_DIR / f"{split}_candidates.parquet", WORK_DIR / f"{split}_s1.parquet",
                       _pool_files(split))
    con = connect(mem_gb=8)
    t = pq.read_table(load_idf(con, split)).to_pydict()
    idf = dict(zip(zip(t["country"], t["tok"]), t["idf"]))
    ids_tbl = pa.table({"s1_id": pa.array(ids, pa.string())})
    con.register("ids", ids_tbl)
    con.execute(f"CREATE OR REPLACE TABLE csub AS SELECT c.* FROM read_parquet('{cand}') c JOIN ids USING (s1_id)")
    con.execute(f"""CREATE OR REPLACE TABLE s1cnt AS SELECT country, name_key, COUNT(*)::INT AS n
        FROM read_parquet('{s1p}') WHERE name_key <> '' GROUP BY 1, 2""")
    con.execute(f"""CREATE OR REPLACE TABLE rival AS
        SELECT c.s1_id, c.cand_id,
               MAX(COALESCE(
                   len(list_intersect(list_distinct(r.addr_tokens), list_distinct(p.addr_tokens)))::DOUBLE
                   / NULLIF(len(list_distinct(list_concat(r.addr_tokens, p.addr_tokens))), 0), 0.0)) AS rival_jacc,
               MAX(CASE WHEN len(list_intersect(r.addr_nums, p.addr_nums)) > 0 THEN 1 ELSE 0 END) AS rival_num
        FROM csub c
        JOIN read_parquet({pool}) p ON p.entity_id = c.cand_id
        JOIN s1cnt k ON k.country = p.country AND k.name_key = p.name_key AND k.n <= 50
        JOIN read_parquet('{s1p}') r ON r.country = p.country AND r.name_key = p.name_key
                                     AND r.entity_id <> c.s1_id
        GROUP BY 1, 2""")
    df = con.execute(f"""
        SELECT c.s1_id, c.cand_id, c.by_a, c.by_a2, c.by_b, c.by_c, c.by_d, c.by_e,
               c.by_f, c.idf_score, c.shared, c.containment, c.rnk,
               s.country s_country, s.name_key s_name_key, s.name_text s_name_text,
               s.name_tokens s_name_tokens, s.name_suffix s_name_suffix,
               s.addr_text s_addr_text, s.addr_tokens s_addr_tokens,
               s.addr_nums s_addr_nums, s.postcode s_postcode, s.addr_empty s_addr_empty,
               p.country c_country, p.name_key c_name_key, p.name_text c_name_text,
               p.name_tokens c_name_tokens, p.name_suffix c_name_suffix,
               p.addr_text c_addr_text, p.addr_tokens c_addr_tokens,
               p.addr_nums c_addr_nums, p.postcode c_postcode, p.addr_empty c_addr_empty,
               COALESCE(ks.n, 1) x_s1_dup, COALESCE(kc.n, 0) x_c_s1cnt,
               rv.rival_jacc x_rival_jacc, rv.rival_num x_rival_num
        FROM csub c
        JOIN read_parquet('{s1p}') s ON s.entity_id = c.s1_id
        JOIN read_parquet({pool}) p ON p.entity_id = c.cand_id
        LEFT JOIN s1cnt ks ON ks.country = s.country AND ks.name_key = s.name_key
        LEFT JOIN s1cnt kc ON kc.country = p.country AND kc.name_key = p.name_key
        LEFT JOIN rival rv ON rv.s1_id = c.s1_id AND rv.cand_id = c.cand_id
        ORDER BY c.s1_id, c.cand_id""").df()
    con.close()
    ref = pair_features(df, idf)
    add_group_features(ref, df["s1_id"].to_numpy())
    ref = pd.DataFrame(ref, columns=FEATURE_COLUMNS).assign(s1_id=df["s1_id"].values,
                                                            cand_id=df["cand_id"].values)
    m = ref.merge(fast, on=["s1_id", "cand_id"], suffixes=("_r", "_f"))
    # the reference predates the G/H flags
    skip = {"b_by_g", "b_by_h", "b_n_schemes"}
    bad = []
    for c in FEATURE_COLUMNS:
        a, b = m[f"{c}_r"].to_numpy(np.float64), m[f"{c}_f"].to_numpy(np.float64)
        same = np.isclose(a, b, atol=1e-3) | (np.isnan(a) & np.isnan(b))
        frac = 1.0 - same.mean()
        if frac > 0:
            print(f"  {c:18s} differs on {frac:.2%}")
        if frac > 0.001 and c not in skip:
            bad.append(c)
    print(f"  checked {len(m):,} pairs of {len(ids)} entities (reference rows {len(ref):,})")
    if bad or len(m) != len(ref):
        raise RuntimeError(f"fast features differ from the reference: {bad}")
    print("  EQUIVALENT")

build_features("train")
check_features("train")

## Stage 4 — expected-F0.5 set selection

In [ ]:
def select_sets(df, prob_col="p", e_m=None):
    d = df[["s1_id", "cand_id", prob_col]].sort_values(
        ["s1_id", prob_col], ascending=[True, False], kind="stable")
    p = d[prob_col].to_numpy(np.float64)
    g = d.groupby("s1_id", sort=False)
    k = g.cumcount().to_numpy() + 1
    cum_p = g[prob_col].cumsum().to_numpy(np.float64)
    e_m_arr = (g[prob_col].transform("sum").to_numpy(np.float64) if e_m is None
               else d["s1_id"].map(e_m).fillna(0.0).to_numpy(np.float64))
    d["score"] = 1.25 * cum_p / (k + 0.25 * e_m_arr)
    d["k"] = k
    d["log1mp"] = np.log1p(-np.clip(p, 0.0, 1.0 - 1e-12))
    best_idx = d.groupby("s1_id", sort=False)["score"].idxmax()
    best = d.loc[best_idx, ["s1_id", "score", "k"]].set_index("s1_id")
    score0 = np.exp(d.groupby("s1_id", sort=False)["log1mp"].sum())
    kstar = best["k"].where(~(score0 > best["score"]), 0)
    d["kstar"] = d["s1_id"].map(kstar).to_numpy()
    sel = d[d["k"] <= d["kstar"]]
    out = {sid: set() for sid in kstar.index}
    for sid, cid in zip(sel["s1_id"].to_numpy(), sel["cand_id"].to_numpy()):
        out[sid].add(cid)
    return out

def select_global_threshold(df, thr, prob_col="p"):
    out = {sid: set() for sid in df["s1_id"].unique()}
    hit = df[df[prob_col] >= thr]
    for sid, cid in zip(hit["s1_id"].to_numpy(), hit["cand_id"].to_numpy()):
        out[sid].add(cid)
    return out

## Stage 5 — global exclusivity repair

In [ ]:
def repair_exclusivity(pred, df, prob_col="p", rounds=3, e_m=None):
    prob = {(s, c): p for s, c, p in zip(df["s1_id"].to_numpy(),
                                        df["cand_id"].to_numpy(), df[prob_col].to_numpy())}
    total = 0
    for _ in range(rounds):
        owners = {}
        for sid, cands in pred.items():
            for cid in cands:
                owners.setdefault(cid, []).append(sid)
        contested = {c: s for c, s in owners.items() if len(s) > 1}
        if not contested:
            break
        total += len(contested)
        drop = set()
        for cid, sids in contested.items():
            winner = max(sids, key=lambda s: prob.get((s, cid), 0.0))
            drop.update((s, cid) for s in sids if s != winner)
        surviving = df[~pd.MultiIndex.from_arrays([df["s1_id"], df["cand_id"]]).isin(drop)]
        pred = select_sets(surviving, prob_col, e_m)
    return pred, total

## Stage 3 — train the pair scorer (70/15/15 by S1 entity), evaluate

In [ ]:
MODEL_PATH, CALIB_PATH = WORK_DIR / "lgbm.txt", WORK_DIR / "calib.npz"

def fold_of(s1_ids):
    h = pd.util.hash_array(np.asarray(s1_ids), hash_key="0" * 16) % 100
    return np.where(h < 70, 0, np.where(h < 85, 1, 2))

def train_model(split="train"):
    import lightgbm as lgb
    from sklearn.isotonic import IsotonicRegression
    from sklearn.metrics import average_precision_score, roc_auc_score

    feats = pd.read_parquet(WORK_DIR / f"{split}_features.parquet")
    truth_all = load_ground_truth(set(feats["s1_id"].unique()))
    pos = {(s, c) for s, cs in truth_all.items() for c in cs}
    feats["y"] = [int((s, c) in pos) for s, c in
                  zip(feats["s1_id"].to_numpy(), feats["cand_id"].to_numpy())]
    feats["fold"] = fold_of(feats["s1_id"].to_numpy())
    print(f"  pairs {len(feats):,}  positives {feats.y.sum():,} ({100*feats.y.mean():.2f}%)")
    tr, ca, va = feats[feats.fold == 0], feats[feats.fold == 1], feats[feats.fold == 2]
    print(f"  entities  train {tr.s1_id.nunique():,}  calib {ca.s1_id.nunique():,}  val {va.s1_id.nunique():,}")

    dtr = lgb.Dataset(tr[FEATURE_COLUMNS], tr["y"])
    dca = lgb.Dataset(ca[FEATURE_COLUMNS], ca["y"], reference=dtr)
    booster = lgb.train(
        dict(objective="binary", metric="average_precision", learning_rate=0.05,
             num_leaves=63, min_data_in_leaf=100, feature_fraction=0.9,
             bagging_fraction=0.8, bagging_freq=1, seed=42, num_threads=CONFIG["threads"],
             verbosity=-1, deterministic=True),
        dtr, num_boost_round=600, valid_sets=[dca],
        callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(200)])
    print(f"  best iteration {booster.best_iteration}")

    raw_ca = booster.predict(ca[FEATURE_COLUMNS], num_iteration=booster.best_iteration)
    iso = IsotonicRegression(out_of_bounds="clip").fit(raw_ca, ca["y"])
    raw_va = booster.predict(va[FEATURE_COLUMNS], num_iteration=booster.best_iteration)
    va = va.copy(); va["p"] = iso.predict(raw_va)
    print(f"  val ROC-AUC {roc_auc_score(va.y, raw_va):.4f}  PR-AUC {average_precision_score(va.y, raw_va):.4f}")
    booster.save_model(str(MODEL_PATH), num_iteration=booster.best_iteration)
    np.savez(CALIB_PATH, x=iso.X_thresholds_, y=iso.y_thresholds_)
    return booster, iso, va

booster, iso, va = train_model()

In [ ]:
truth = load_ground_truth(set(va["s1_id"].unique()))
best_thr, best_score = 0.5, -1.0
for thr in np.arange(0.05, 0.96, 0.05):
    s = macro_f05(select_global_threshold(va, thr), truth)
    if s > best_score:
        best_thr, best_score = thr, s
sel = select_sets(va)
exp_score = macro_f05(sel, truth)
repaired, conflicts = repair_exclusivity(sel, va)
rep_score = macro_f05(repaired, truth)

print("=== VAL MACRO F_0.5 ===")
print(f"  global threshold (best {best_thr:.2f})   {best_score:.4f}")
print(f"  expected-F_0.5 selection          {exp_score:.4f}  ({exp_score-best_score:+.4f})")
print(f"  + exclusivity repair              {rep_score:.4f}  ({rep_score-exp_score:+.4f}, {conflicts:,} contested)")
imp = pd.Series(booster.feature_importance("gain"), index=FEATURE_COLUMNS).sort_values(ascending=False)
print("\n=== TOP FEATURES (gain) ===")
for k, v in imp.head(15).items():
    print(f"  {k:<18s} {v:12,.0f}")

In [ ]:
# ---- error analysis (read-only) ----
cset = va.groupby("s1_id")["cand_id"].apply(set).to_dict()
s1meta = pd.read_parquet(WORK_DIR / "train_s1.parquet", columns=["entity_id", "country"],
                         filters=[("entity_id", "in", list(truth))]).set_index("entity_id")
rows = []
for sid, t in truth.items():
    p = repaired.get(sid, set()); c = cset.get(sid, set())
    m, k, h, found = len(t), len(p), len(p & t), len(t & c)
    if m == 0:        cat = "singleton OK" if k == 0 else "singleton given a match"
    elif found == 0:  cat = "blocking found none"
    elif k == 0:      cat = "model predicted nothing"
    elif k > h:       cat = "has wrong match(es)"
    elif h < m:       cat = "correct but incomplete"
    else:             cat = "perfect"
    rows.append((sid, cat, 1 - f05(p, t), s1meta["country"].get(sid, "?")))
E = pd.DataFrame(rows, columns=["s1_id", "cat", "loss", "country"])
t = E.groupby("cat").agg(entities=("s1_id", "size"), score_lost=("loss", "sum"))
t["score_lost"] /= len(E)
print(t.sort_values("score_lost", ascending=False).to_string(float_format=lambda x: f"{x:.4f}"))
print(E.groupby("country").agg(entities=("s1_id", "size"),
      macro_f=("loss", lambda x: 1 - x.mean())).to_string(float_format=lambda x: f"{x:.4f}"))

## Test inference + submission (runs only when RUN_TEST = True)

In [ ]:
PROB_FLOOR = 0.01

def predict_shard(split, suffix, booster, iso):
    feats = pd.read_parquet(WORK_DIR / f"{split}_features{suffix}.parquet")
    p = iso.predict(booster.predict(feats[FEATURE_COLUMNS]))
    out = pd.DataFrame({"s1_id": feats.s1_id, "cand_id": feats.cand_id, "p": p})
    sums = out.groupby("s1_id", sort=False)["p"].sum().rename("e_m").reset_index()
    kept = out[out.p >= PROB_FLOOR]
    kept.to_parquet(WORK_DIR / f"{split}_probs{suffix}.parquet", index=False)
    sums.to_parquet(WORK_DIR / f"{split}_sums{suffix}.parquet", index=False)
    print(f"  {suffix}: scored {len(out):,} -> kept {len(kept):,}, {len(sums):,} entities")

def run_test(shards=None, resume=True):
    shards = shards or CONFIG["test_shards"]
    if not resume:
        for pat in ("test_probs*", "test_sums*", "test_candidates*"):
            for f in WORK_DIR.glob(f"{pat}.parquet"):
                f.unlink()
    for f in WORK_DIR.glob("test_features*.parquet"):
        f.unlink()

    def done(sfx):
        return all((WORK_DIR / f"test_{k}{sfx}.parquet").exists() for k in ("probs", "sums", "candidates"))

    todo = [i for i in range(shards) if not done(f"_sh{i:02d}")]
    if not todo:
        print("all shards already complete - go straight to assemble()")
        return
    print(f"shards to run: {todo}")
    dbfile = WORK_DIR / "er_test.duckdb"
    for suf in ("", ".wal"):
        Path(f"{dbfile}{suf}").unlink(missing_ok=True)
    con = connect(db=dbfile)
    print("--- pool index (once, on disk) ---"); build_pool(con, "test")
    for i in todo:
        sfx = f"_sh{i:02d}"
        t_shard = time.perf_counter()
        print(f"\n--- shard {i+1}/{shards} ---")
        build_shard(con, "test", f"WHERE hash(entity_id) % {shards} = {i}", sfx)
        for t in ("upairs", "candidates", "s1_rare", "s1_skel", "s1_arare", "s1_rare4", "s1_pair",
                  "s1_g", "s1_h", "s1_norm", "pairs_a", "pairs_a2", "pairs_b", "pairs_c",
                  "pairs_d", "pairs_e", "pairs_f", "pairs_g", "pairs_h"):
            con.execute(f"DROP TABLE IF EXISTS {t}")
        con.execute("CHECKPOINT")
        build_features("test", sfx)
        predict_shard("test", sfx, booster, iso)
        (WORK_DIR / f"test_features{sfx}.parquet").unlink(missing_ok=True)
        cpath = WORK_DIR / f"test_candidates{sfx}.parquet"
        pq.write_table(pq.read_table(cpath, columns=["s1_id", "cand_id"]), cpath, compression="zstd")
        print(f"  shard {time.perf_counter() - t_shard:.0f}s  free disk {shutil.disk_usage(WORK_DIR).free/2**30:.1f} GB")
    con.close()

def assemble(split="test"):
    probs = pd.concat([pd.read_parquet(f) for f in
                       sorted(WORK_DIR.glob(f"{split}_probs*.parquet"))], ignore_index=True)
    sums = pd.concat([pd.read_parquet(f) for f in sorted(WORK_DIR.glob(f"{split}_sums*.parquet"))],
                     ignore_index=True).groupby("s1_id", sort=False)["e_m"].sum()
    print(f"  probs {len(probs):,}  entities with candidates {len(sums):,}")
    pred = select_sets(probs, "p", sums)
    pred, conflicts = repair_exclusivity(pred, probs, "p", 3, sums)
    print(f"  contested ids resolved: {conflicts:,}")

    s1_path = WORK_DIR / f"{split}_s1.parquet"
    all_s1 = pq.read_table(s1_path, columns=["entity_id"])["entity_id"].to_pylist()
    with open(OUTPUT_DIR / "matching_results.tsv", "w") as fh:
        fh.write("source1_entity_id\tmatched_entity_ids\n")
        for sid in all_s1:
            fh.write(f"{sid}\t{','.join(sorted(pred.get(sid, ())))}\n")

    # candidate_pairs.tsv via DuckDB: ~170M pairs would not fit in a Python dict
    cand_files = [str(f) for f in sorted(WORK_DIR.glob(f"{split}_candidates*.parquet"))]
    con = connect(mem_gb=12)
    con.execute(f"""COPY (
        SELECT s.entity_id AS source1_entity_id,
               string_agg(c.cand_id, ',' ORDER BY c.cand_id) AS candidate_entity_ids
        FROM read_parquet('{s1_path}', file_row_number = true) s
        LEFT JOIN (SELECT s1_id, cand_id FROM read_parquet({cand_files})) c ON c.s1_id = s.entity_id
        GROUP BY s.entity_id, s.file_row_number
        ORDER BY s.file_row_number
    ) TO '{OUTPUT_DIR / "candidate_pairs.tsv"}' (FORMAT CSV, DELIMITER '\t', HEADER)""")
    con.close()
    n = sum(1 for s in all_s1 if pred.get(s))
    print(f"  wrote both TSVs: {len(all_s1):,} rows, {n:,} with >=1 match ({100*n/len(all_s1):.1f}%)")

if RUN_TEST:
    run_test()
    assemble()

## Validate before uploading (must print PASS)

In [ ]:
if RUN_TEST:
    validator = DATA_DIR.parent / "validate_submission.py"
    cmd = [sys.executable, str(validator),
           "--matching", str(OUTPUT_DIR / "matching_results.tsv"),
           "--candidate", str(OUTPUT_DIR / "candidate_pairs.tsv"),
           "--test-dir", str(DATA_DIR / "test")]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:] or r.stderr[-2000:])
    print(f"exit code {r.returncode}  ->  {'PASS' if r.returncode == 0 else 'FAIL'}")